In [1]:
from pathlib import Path
import os
from sqlalchemy import Column, Date, Float, Integer, Boolean, MetaData, String, Table, create_engine, text
import pandas as pd


In [17]:
print("starting data cleaning Northwind_error.csv")
print(f"{Path.cwd()=}")

# check data is available where expected and encoding matches
with open("data/Northwind_errors.csv", "r", encoding='ISO-8859-1') as file:
    data = file.readlines()
    print("Data from nortNorthwind_errorshwind.csv:")
    print(data[:2])
    print(data[5])  # special encoding

starting data cleaning Northwind_error.csv
Path.cwd()=PosixPath('/home/markus/data-eng-neue-fische/data-eng-exercises/week4-data-processing/day4-data-cleaning')
Data from nortNorthwind_errorshwind.csv:
['orderID,customerID,employeeID,orderDate,requiredDate,shippedDate,shipVia,Freight,productID,unitPrice,quantity,discount,companyName,contactName,contactTitle,employees.lastName,employees.firstName,employees.title,productName,supplierID,categoryID,quantityPerUnit,unitPrice.1,unitsInStock,unitsOnOrder,reorderLevel,discontinued,categoryName,suppliers.companyName,suppliers.contactName,suppliers.contactTitle\n', ",VINET,5,1996-07-04,1996-08-01,1996-07-16,3,32.38,11,14,-12,0,Vins et alcools Chevalier,Paul Henriot,Accounting Manager,Buchanan,Steven,Sales Manager,Queso Cabrales,5,4,1 kg pkg.,21,22,30,30,0,Dairy Products,Cooperativa de Quesos 'Las Cabras',Antonio del Valle Saavedra,Export Administrator\n"]
10264,FOLKO,6,1996-07-24,1996-08-21,1996-08-23,3,3.67,2,15.2,35,0,Folk och fä HB,Maria Lars

In [39]:
PG_USER="postgres"
PG_PASSWORD="postgres"
DB_NAME="db"
PG_HOST="localhost"
PG_PORT="5433"

DATABASE_URL = f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{DB_NAME}"

engine = create_engine(DATABASE_URL)
metadata = MetaData(schema="public")  # optional schema, default is 'public'


In [40]:
NAME_TBL_NORTHWIND = "northwindtbl"

In [41]:
with engine.begin() as conn:
    conn.execute(text(f"DROP TABLE IF EXISTS {NAME_TBL_NORTHWIND}"))

In [42]:
with engine.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
    # Check if the database exists
    result = conn.execute(
        text(f"SELECT 1 FROM pg_database WHERE datname = '{DB_NAME}'"))
    exists = result.scalar()
    print(f"Database '{DB_NAME}' exists: {exists}")

Database 'db' exists: 1


In [43]:

"""
orderID,customerID,employeeID,orderDate,requiredDate,shippedDate,shipVia,Freight,
productID,unitPrice,quantity,discount,companyName,contactName,contactTitle,
employees.lastName,employees.firstName,employees.title,productName,supplierID,
categoryID,quantityPerUnit,unitPrice.1,unitsInStock,unitsOnOrder,reorderLevel,discontinued,
categoryName,suppliers.companyName,suppliers.contactName,suppliers.contactTitle
"""
northwindtbl = Table(
    NAME_TBL_NORTHWIND,
    metadata,
    Column("orderID", Integer, primary_key=True),
    Column("customerID", String),
    Column("employeeID", Integer),
    Column("orderDate", Date),
    Column("requiredDate", Date),
    Column("shippedDate", Date),
    Column("shipVia", Integer),
    Column("freight", Float),
    Column("productID", Integer),
    Column("unitPriceAsOrdered", Float),
    Column("quantity", Integer),
    Column("discount", Float),
    Column("companyName", String),
    Column("contactName", String),
    Column("contactTitle", String),
    Column("employeesLastName", String),
    Column("employeesFirstName", String),
    Column("employeesTitle", String),
    Column("productName", String),
    Column("supplierID", Integer),
    Column("categoryID", Integer),
    Column("quantityPerUnit", String),
    Column("unitPriceCurrent", Float),
    Column("unitsInStock", Integer),
    Column("unitsOnOrder", Integer),
    Column("reorderLevel", Integer),
    Column("discontinued", Boolean),
    Column("categoryName", String),
    Column("suppliersCompanyName", String),
    Column("suppliersContactName", String),
    Column("suppliersContactTitle", String),
)

metadata.create_all(engine, checkfirst=True)
print(f"[INFO] Table '{NAME_TBL_NORTHWIND}' created or already exists.")


[INFO] Table 'northwindtbl' created or already exists.


## play around with metadata object and learn what it has

In [44]:
dir(metadata)[-12:]

['clear',
 'create_all',
 'create_drop_stringify_dialect',
 'dispatch',
 'drop_all',
 'info',
 'naming_convention',
 'reflect',
 'remove',
 'schema',
 'sorted_tables',
 'tables']

In [61]:
metadata.schema

'public'

In [63]:
metadata.info

{}

In [59]:
metadata.tables

FacadeDict({'public.northwindtbl': Table('northwindtbl', MetaData(), Column('orderID', Integer(), table=<northwindtbl>, primary_key=True, nullable=False), Column('customerID', String(), table=<northwindtbl>), Column('employeeID', Integer(), table=<northwindtbl>), Column('orderDate', Date(), table=<northwindtbl>), Column('requiredDate', Date(), table=<northwindtbl>), Column('shippedDate', Date(), table=<northwindtbl>), Column('shipVia', Integer(), table=<northwindtbl>), Column('freight', Float(), table=<northwindtbl>), Column('productID', Integer(), table=<northwindtbl>), Column('unitPriceAsOrdered', Float(), table=<northwindtbl>), Column('quantity', Integer(), table=<northwindtbl>), Column('discount', Float(), table=<northwindtbl>), Column('companyName', String(), table=<northwindtbl>), Column('contactName', String(), table=<northwindtbl>), Column('contactTitle', String(), table=<northwindtbl>), Column('employeesLastName', String(), table=<northwindtbl>), Column('employeesFirstName', S

In [68]:
with engine.connect() as conn:
    res = conn.execute(text(f"SELECT 1"))
print(res.fetchone())

(1,)


In [69]:
col_names = northwindtbl.columns.keys()

In [70]:
df = pd.read_csv("data/Northwind_errors.csv", skiprows=1, names=col_names, encoding='ISO-8859-1')
print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 31 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   orderID                996 non-null    float64
 1   customerID             998 non-null    str    
 2   employeeID             1000 non-null   int64  
 3   orderDate              1000 non-null   str    
 4   requiredDate           1000 non-null   str    
 5   shippedDate            1000 non-null   str    
 6   shipVia                1000 non-null   int64  
 7   freight                1000 non-null   float64
 8   productID              1000 non-null   int64  
 9   unitPriceAsOrdered     1000 non-null   float64
 10  quantity               1000 non-null   int64  
 11  discount               1000 non-null   float64
 12  companyName            1000 non-null   str    
 13  contactName            1000 non-null   str    
 14  contactTitle           1000 non-null   str    
 15  employeesLastNam

In [72]:
df.to_sql(NAME_TBL_NORTHWIND, con=engine, if_exists="replace", index=False)

1000

## IMPORTANT: make sure to quote any names conataining capital letter like "orderID" because SQL conversts everything else to lower case by default!!!

In [73]:
df1 = pd.read_sql(f"""SELECT "orderID", "orderDate"  FROM {NAME_TBL_NORTHWIND}""", con=engine)
print(df1.info())
print(df1.head())

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   orderID    996 non-null    float64
 1   orderDate  1000 non-null   str    
dtypes: float64(1), str(1)
memory usage: 25.5 KB
None
   orderID   orderDate
0      NaN  1996-07-04
1  10266.0  1996-07-26
2  10258.0  1996-07-17
3  10255.0  1996-07-12
4  10264.0  1996-07-24


In [78]:
with engine.connect() as conn:
    res = conn.execute(text(f"SELECT * FROM {NAME_TBL_NORTHWIND}")).fetchmany(3)
for r in res:
    print(r)

(None, 'VINET', 5, '1996-07-04', '1996-08-01', '1996-07-16', 3, 32.38, 11, 14.0, -12, 0.0, 'Vins et alcools Chevalier', 'Paul Henriot', 'Accounting Manager', 'Buchanan', 'Steven', 'Sales Manager', 'Queso Cabrales', 5, 4, '1 kg pkg.', 21.0, 22, 30, 30, 0, 'Dairy Products', "Cooperativa de Quesos 'Las Cabras'", 'Antonio del Valle Saavedra', 'Export Administrator')
(10266.0, 'WARTH', 3, '1996-07-26', '1996-09-06', '1996-07-31', 3, 25.73, 12, 30.4, -12, 0.05, 'Wartian Herkku', 'Pirkko Koskitalo', 'Accounting Manager', 'Leverling', 'Janet', 'Sales Representative', 'Queso Manchego La Pastora', 5, 4, '10 - 500 g pkgs.', 38.0, 86, 0, 0, 0, 'Dairy Products', "Cooperativa de Quesos 'Las Cabras'", 'Antonio del Valle Saavedra', 'Export Administrator')
(10258.0, 'ERNSH', 1, '1996-07-17', '1996-08-14', '1996-07-23', 1, 140.51, 2, 15.2, -50, 0.2, 'Ernst Handel', 'Roland Mendel', 'Sales Manager', 'Davolio', 'Nancy', 'Sales Representative', 'Chang', 1, 1, '24 - 12 oz bottles', 19.0, 17, 40, 25, 0, 'Bev

In [ ]:
print("data cleaning Northwind_error.csv finished")